Read Silver Table

In [0]:
# df = spark.table(
#     "weather_catalog.gold.location_master"
# )

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:440)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:80)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:348)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Attr

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04_gold_to_azure_sql_reporting
# MAGIC **Purpose:** Read curated data from the Gold layer (Unity Catalog) and write it to the Azure SQL Database reporting instance.

# COMMAND ----------
# 1. Configuration & Security Setup
# Replace 'weather-scope' with your secret scope name, or use string variables if not using secrets yet.

jdbc_hostname = "weather-sql-server-new.database.windows.net" 
jdbc_port     = "1433"
jdbc_database = "db_name"
jdbc_username = ""
#jdbc_password = dbutils.secrets.get(scope="weather-scope", key="sql-password") 
jdbc_password = ""

# Build the JDBC Connection URL
jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};database={jdbc_database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

# Target Configuration
gold_table_identifier = "weather_catalog.gold.location_master"
target_sql_table      = "dbo.location_master"

# COMMAND ----------
# 2. Read Curated Data from Unity Catalog Gold Layer
print(f"Reading data from Gold Table: {gold_table_identifier}")
gold_df = spark.table(gold_table_identifier)

# COMMAND ----------
# 3. Optional: Schema Optimization for SQL Database
# Azure SQL strings are ideally mapped to specific lengths. 
# You can cast columns here if your target SQL DDL strictly requires VARCHAR(X) instead of NVARCHAR(MAX).
reporting_df = gold_df

# COMMAND ----------
# 4. Write Data to Azure SQL Database
print(f"Writing data to Azure SQL Table: {target_sql_table}")

try:
    reporting_df.write \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", target_sql_table) \
        .option("user", jdbc_username) \
        .option("password", jdbc_password) \
        .mode("overwrite") \
        .save()
    
    print("--- Data Transfer to Azure SQL Successfully Completed ---")
    print(f"Total Rows Synchronized: {reporting_df.count()}")

except Exception as e:
    print(f"CRITICAL ERROR during SQL Sync: {str(e)}")
    raise e

Reading data from Gold Table: weather_catalog.gold.location_master
Writing data to Azure SQL Table: dbo.location_master
--- Data Transfer to Azure SQL Successfully Completed ---
Total Rows Synchronized: 3
